# 1.2 · Architecture: hosts, clients and servers

**架构：宿主、客户端与服务器**

> **After this lesson you can:** Draw MCP's host–client–server architecture and its two layers from memory, trace a full discover → list → call → notify exchange, and explain how the architecture isolates servers from each other.

Official reading: [Architecture overview](https://modelcontextprotocol.io/docs/2026-07-28/learn/architecture) · [Spec: Architecture](https://modelcontextprotocol.io/specification/2026-07-28/architecture/index)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. Two focused servers
The spec wants servers that are *extremely easy to build* and *focused*. Here are two: a project-files server and a ticket tracker. Each records, in its own audit log, exactly what it received — so we can later prove what each server could see.

In [ ]:
import json
from contextlib import AsyncExitStack

from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context


def show(obj):
    print(json.dumps(obj, indent=2, default=str))


# --- server 1: project files -------------------------------------------------
files_server = MCPServer("files", version="1.0.0")
FILES = {"notes.txt": "Ship v2 on Friday.", "secret.env": "API_KEY=sk-live-123"}
files_audit = []


@files_server.tool()
def read_file(path: str, ctx: Context) -> str:
    """Read a file from the project folder."""
    files_audit.append({"request": ctx.request_id, "arguments": {"path": path}})
    return FILES[path]


# --- server 2: ticket tracker ------------------------------------------------
tickets_server = MCPServer("tickets", version="0.3.0")
tickets_audit = []


@tickets_server.tool()
def create_ticket(title: str, ctx: Context) -> str:
    """Create a ticket in the issue tracker and return its id."""
    caps = ctx.client_capabilities
    tickets_audit.append({"request": ctx.request_id, "arguments": {"title": title},
                          "protocolVersion": ctx.protocol_version,
                          "clientCapabilities": caps.model_dump(by_alias=True, exclude_none=True) if caps else None})
    return f"T-{len(tickets_audit)}"


print("servers:", files_server.name, "and", tickets_server.name)

### 2. A host creates one client per server
Our `Host` owns everything the spec gives the host: the client instances, the merged tool registry, the conversation. Each `Client` discovers its server while connecting, so `protocol_version`, `server_info` and `server_capabilities` are ready afterwards.

> **SDK note.** Enter the clients with an `AsyncExitStack`: they must be closed in reverse order of opening (anyio cancel scopes). Calling `__aexit__` on them in any other order raises `RuntimeError`.

In [ ]:
class Host:
    """A miniature MCP host: one client per server, one tool registry, one conversation."""

    def __init__(self, servers):
        self.servers = servers
        self.clients = {}          # server name -> Client
        self.registry = {}         # tool name exposed to the model -> (server name, real tool name)
        self.conversation = []     # stays in the host; servers never see it

    async def __aenter__(self):
        self._stack = AsyncExitStack()
        for server in self.servers:
            client = await self._stack.enter_async_context(Client(server))
            self.clients[client.server_info.name] = client
        await self.refresh_tools()
        return self

    async def __aexit__(self, *exc):
        await self._stack.aclose()

    async def refresh_tools(self):
        self.registry.clear()
        for server_name, client in self.clients.items():
            for tool in (await client.list_tools()).tools:
                exposed = tool.name
                if exposed in self.registry:                    # two servers, same tool name
                    exposed = f"{server_name}__{tool.name}"     # namespace it for the model
                self.registry[exposed] = (server_name, tool.name)

    async def call(self, exposed_name, arguments):
        server_name, tool_name = self.registry[exposed_name]  # route by name
        result = await self.clients[server_name].call_tool(tool_name, arguments)
        text = result.content[0].text
        self.conversation.append({"tool": exposed_name, "server": server_name, "result": text})
        return text


async with Host([files_server, tickets_server]) as host:
    for name, client in host.clients.items():
        print(f"client -> {name} v{client.server_info.version}  protocol={client.protocol_version}")
        print("   server capabilities:", client.server_capabilities.model_dump(by_alias=True, exclude_none=True))
    print("\nunified tool registry the model would see:")
    show(host.registry)

### 3. Route tool calls by name
This is the docs' `handle_tool_call` pseudo-code made real: the "model" names a tool, the host finds the owning client, calls it, and appends the result to the conversation.

In [ ]:
files_audit.clear(); tickets_audit.clear()

async with Host([files_server, tickets_server]) as host:
    # pretend the model asked for these two calls, in order
    note = await host.call("read_file", {"path": "notes.txt"})
    secret = await host.call("read_file", {"path": "secret.env"})
    ticket = await host.call("create_ticket", {"title": f"Follow up: {note}"})
    print("host conversation (lives only in the host):")
    show(host.conversation)

### 4. Isolation: what did each server actually see?
Compare the host's conversation with each server's audit log. The tickets server saw one title — never the secret the files server returned, never the conversation. Data crossed from one server to another **only** because the host chose to put `note` into the ticket title.

In [ ]:
print("files server saw:")
show(files_audit)
print("\ntickets server saw:")
show(tickets_audit)

leaked = any("sk-live" in json.dumps(entry) for entry in tickets_audit)
print("\ndid the secret reach the tickets server?", leaked)

Now imagine the model had been prompt-injected into calling `create_ticket` with the secret in the title. The protocol would carry it faithfully — the boundary is only as good as the host's policy. A real host would put a consent gate in `Host.call`.

### 5. Capabilities travel with every request
In 2026-07-28 the client declares its capabilities on *each* request. The same server sees different capabilities from a client that can handle elicitation and one that cannot — with no handshake in between.

In [ ]:
async def handle_elicitation(context, params):
    return None   # never invoked here; its presence makes the client advertise `elicitation`


tickets_audit.clear()
async with Client(tickets_server) as plain:
    await plain.call_tool("create_ticket", {"title": "from a plain client"})
async with Client(tickets_server, elicitation_callback=handle_elicitation) as rich:
    await rich.call_tool("create_ticket", {"title": "from an elicitation-capable client"})

for entry in tickets_audit:
    print(f"{entry['arguments']['title']:<38} -> clientCapabilities = {entry['clientCapabilities']}")

### 6. Name collisions are the host's problem
Tool names are unique only *within* a server. Add a second server that also has `read_file`, and the host must namespace it before showing tools to the model.

In [ ]:
wiki_server = MCPServer("wiki", version="2.0.0")


@wiki_server.tool()
def read_file(path: str) -> str:
    """Read a page from the team wiki."""
    return f"wiki page {path}"


async with Host([files_server, tickets_server, wiki_server]) as host:
    show(host.registry)
    print(await host.call("wiki__read_file", {"path": "onboarding"}))

### 7. The docs' worked example, on the wire
Now the full sequence from the architecture page — discover → `tools/list` → `tools/call` → `subscriptions/listen` → `list_changed` → refresh — against a real stdio subprocess, with no SDK on the client side. The `enable_forecast` tool adds a tool at runtime, so the server has a reason to notify.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "weather_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    server = MCPServer("weather", version="1.0.0")

    @server.tool()
    def weather_current(location: str, units: str = "metric") -> str:
        """Get current weather information for a location."""
        return f"{location}: 20 degrees ({units}), partly cloudy"

    @server.tool()
    async def enable_forecast(ctx: Context) -> str:
        """Unlock the multi-day forecast tool."""
        def weather_forecast(location: str, days: int = 3) -> str:
            """Get a multi-day forecast."""
            return f"{location}: sunny for {days} days"
        server.add_tool(weather_forecast)
        await ctx.notify_tools_changed()
        return "forecast enabled"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "weather_server.py")],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientInfo": {"name": "example-client", "version": "1.0.0"},
        "io.modelcontextprotocol/clientCapabilities": {}}


def send(message):
    print(">>>", json.dumps(message)[:220])
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()


def receive_until(request_id):
    """Read lines until the response to `request_id`; notifications may arrive first."""
    while True:
        message = json.loads(proc.stdout.readline())
        kind = "notification" if "id" not in message else "response"
        print(f"<<< ({kind})", json.dumps(message)[:300])
        if message.get("id") == request_id:
            return message


print("--- step 1: discovery")
send({"jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {"_meta": META}})
discovery = receive_until(1)
print("\nfull discovery result:")
show(discovery["result"])

In [ ]:
print("--- step 2: tool discovery")
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {"_meta": META}})
listing = receive_until(2)
print("tools:", [t["name"] for t in listing["result"]["tools"]])

print("\n--- step 3: tool execution")
send({"jsonrpc": "2.0", "id": 3, "method": "tools/call",
      "params": {"name": "weather_current", "arguments": {"location": "San Francisco", "units": "imperial"},
                 "_meta": META}})
call_reply = receive_until(3)

In [ ]:
print("--- step 4: opt in to change notifications (the request id becomes the subscriptionId)")
send({"jsonrpc": "2.0", "id": 4, "method": "subscriptions/listen",
      "params": {"_meta": META, "notifications": {"toolsListChanged": True}}})
print("<<< (notification)", proc.stdout.readline().strip())   # the acknowledgement comes first

print("\n--- something changes the server's tools")
send({"jsonrpc": "2.0", "id": 5, "method": "tools/call",
      "params": {"name": "enable_forecast", "arguments": {}, "_meta": META}})
enable_reply = receive_until(5)

print("\n--- the client reacts by refreshing the list")
send({"jsonrpc": "2.0", "id": 6, "method": "tools/list", "params": {"_meta": META}})
print("tools now:", [t["name"] for t in receive_until(6)["result"]["tools"]])

proc.stdin.close()
print("server exit code:", proc.wait(timeout=10))

Look at the order: `notifications/tools/list_changed` (no `id`, tagged `subscriptionId: 4`) arrived *before* the response to request 5. Notifications are independent messages; a client must be ready for them at any time on the subscription stream.

### Your turn
1. In section 3, add a **consent gate** to `Host.call`: refuse (and record a refusal in the conversation) any call to a tool on the `tickets` server whose arguments contain `"API_KEY"`. Re-run sections 3–4 with a call that tries to file the secret as a ticket title.
2. In section 7, send a `subscriptions/listen` request that asks for `{"toolsListChanged": True, "promptsListChanged": True}` and read the acknowledgement. Which part of the filter did the server agree to honour, and how would a client know?
3. In section 7, send a `tools/call` whose `_meta` declares `"io.modelcontextprotocol/clientCapabilities": {"elicitation": {}}` and one without, to the same process. Explain why this is legal in 2026-07-28 but would have been meaningless in a 2025-11-25 session.
4. Sketch (in a markdown cell) how your `Host` would change if `files_server` ran remotely over Streamable HTTP. Which parts of the class change, and which stay exactly the same? Why?